# Statistical Inference

## Overview

## Purpose

The purpose of this section is to determine whether the customer behaviour
patterns observed during descriptive analysis are statistically significant.

Average Order Value (AOV) is used as the main customer-value measure because
it represents the typical value generated per order without being mechanically
increased simply because a customer places more orders.

The original AverageOrderValue variable was extremely right-skewed.
Therefore, the log-transformed variable `Log_AverageOrderValue` is used for
parametric statistical tests.

The original AverageOrderValue values are retained for practical business
interpretation.

Two main analyses are performed:

1. Comparison of mean customer value between one-time and repeat customers.
2. Comparison of customer value across low-, medium-, and high-frequency
   customer groups.

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES AND LOAD FINAL CUSTOMER-LEVEL DATA
# ============================================================

# pandas is used for working with the customer-level dataset.
import pandas as pd

# numpy is used for numerical operations and for creating customer groups.
import numpy as np

# scipy provides the statistical tests required in this section.
from scipy.stats import levene, ttest_ind

from pathlib import Path

base_path = Path("../data/processed")

# We use customer-level data because the research questions in Task 4
# compare CUSTOMER behaviour.
# One row in this dataset represents one customer.
df_customers = pd.read_csv(
    base_path / "customer_level_data.csv"
)

# Confirm that the dataset loaded correctly.
print("Dataset shape:", df_customers.shape)

# Confirm that the variables needed for statistical inference exist.
print(df_customers.columns.tolist())

## 4.1 Comparison of Mean Customer Value

Customers are divided into two naturally interpretable groups:

- **One-time customers:** made exactly one purchase.
- **Repeat customers:** made more than one purchase.

This grouping is based directly on observed purchase behaviour rather than
an arbitrary spending threshold.

The analysis investigates whether these two customer groups differ in their
Average Order Value.

In [ ]:
# ============================================================
# 2. CREATE ONE-TIME AND REPEAT CUSTOMER GROUPS
# ============================================================

# Frequency represents the number of orders made by each customer.
#
# Frequency = 1  -> customer purchased only once.
# Frequency > 1 -> customer returned and purchased again.
#
# This creates a simple and business-relevant grouping that directly
# relates to the project's repeat-purchase problem.

df_customers["CustomerGroup"] = np.where(
    df_customers["Frequency"] == 1,
    "One-time",
    "Repeat"
)

# Check how many customers belong to each group.
customer_group_counts = df_customers["CustomerGroup"].value_counts()

print(customer_group_counts)

In [ ]:
# ============================================================
# 3. DESCRIPTIVE AOV VALUES FOR BUSINESS INTERPRETATION
# ============================================================

# Statistical testing will use Log_AverageOrderValue because the original
# AOV distribution was extremely right-skewed.
#
# However, the original AOV is shown here because actual monetary values
# are easier for managers to understand.

aov_summary = (
    df_customers
    .groupby("CustomerGroup")["AverageOrderValue"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

print(aov_summary)

In [ ]:
# ============================================================
# 4. PREPARE LOG-AOV VALUES FOR THE TWO CUSTOMER GROUPS
# ============================================================

# The original AverageOrderValue had extreme positive skewness.
#
# During feature engineering, log(1 + AOV) reduced this skewness
# substantially.
#
# Therefore, Log_AverageOrderValue is more suitable for the
# parametric comparison of means.

one_time_log_aov = df_customers.loc[
    df_customers["CustomerGroup"] == "One-time",
    "Log_AverageOrderValue"
]

repeat_log_aov = df_customers.loc[
    df_customers["CustomerGroup"] == "Repeat",
    "Log_AverageOrderValue"
]

# Confirm the number of observations in each independent group.
print("One-time customers:", len(one_time_log_aov))
print("Repeat customers:", len(repeat_log_aov))

# Also display the mean log-AOV in each group.
print("\nMean Log-AOV:")
print(
    df_customers.groupby("CustomerGroup")[
        "Log_AverageOrderValue"
    ].mean()
)

### Hypotheses

**Null hypothesis (H₀):**

The mean log-transformed Average Order Value is the same for one-time and
repeat customers.

\[
H_0:\mu_{One-time}=\mu_{Repeat}
\]

**Alternative hypothesis (H₁):**

The mean log-transformed Average Order Value differs between one-time and
repeat customers.

\[
H_1:\mu_{One-time}\neq\mu_{Repeat}
\]

A significance level of 5% (\(\alpha = 0.05\)) is used.

In [ ]:
# ============================================================
# 5. CHECK EQUALITY OF VARIANCES USING LEVENE'S TEST
# ============================================================

# An ordinary independent-samples t-test assumes that the two groups
# have approximately equal variances.
#
# Levene's test checks this assumption.
#
# H0: The two groups have equal variances.
# H1: The group variances are different.
#
# If p > 0.05:
#     There is no strong evidence of unequal variance.
#     Standard independent t-test may be used.
#
# If p < 0.05:
#     Equal-variance assumption is violated.
#     Welch's independent t-test should be used instead.

levene_stat, levene_p = levene(
    one_time_log_aov,
    repeat_log_aov
)

print("Levene statistic:", levene_stat)
print("Levene p-value:", levene_p)

In [ ]:
# ============================================================
# 6. COMPARE MEAN CUSTOMER VALUE USING WELCH'S T-TEST
# ============================================================

# Levene's test showed that the variances of the two groups are
# significantly different.
#
# Therefore, Welch's t-test is used instead of the standard
# independent-samples t-test.
#
# Welch's t-test compares the mean Log_AverageOrderValue of
# one-time and repeat customers without assuming equal variances.

from scipy.stats import ttest_ind

t_stat, t_p = ttest_ind(
    one_time_log_aov,
    repeat_log_aov,
    equal_var=False
)

print("Welch t-statistic:", t_stat)
print("p-value:", t_p)

In [ ]:
# Mean Log-AOV for each customer group
mean_log_aov = (
    df_customers
    .groupby("CustomerGroup")["Log_AverageOrderValue"]
    .mean()
)

print("\nMean Log-Average Order Value:")
print(mean_log_aov)

In [ ]:
# Original AOV values are easier to explain to the business.
business_aov_summary = (
    df_customers
    .groupby("CustomerGroup")["AverageOrderValue"]
    .agg(["mean", "median"])
    .round(2)
)

print("\nOriginal AOV summary:")
print(business_aov_summary)

## 4.2 Customer Value Across Purchase-Frequency Levels

The previous analysis compared one-time and repeat customers.

To examine customer value in more detail, customers are now divided into
three purchase-frequency groups:

- Low frequency: 1 order
- Medium frequency: 2–7 orders
- High frequency: 8 or more orders

The purpose is to test whether Average Order Value changes as customer
purchase frequency increases.

`Log_AverageOrderValue` is used for statistical testing because the original
AOV distribution was extremely right-skewed.

In [ ]:
# ============================================================
# 7. CREATE LOW, MEDIUM, AND HIGH FREQUENCY GROUPS
# ============================================================

# Frequency is used to represent how actively a customer purchases.
#
# The grouping is:
# 1 order      -> Low
# 2 to 7       -> Medium
# 8 or more    -> High
#
# The upper cut-off of 7 is consistent with the observed Frequency
# distribution, where the third quartile was approximately 7.

def create_frequency_group(frequency):
    if frequency == 1:
        return "Low"
    elif frequency <= 7:
        return "Medium"
    else:
        return "High"

df_customers["FrequencyGroup"] = (
    df_customers["Frequency"]
    .apply(create_frequency_group)
)

# Check the number of customers in each group.
print(df_customers["FrequencyGroup"].value_counts())

In [ ]:
# ============================================================
# 8. DESCRIPTIVE AOV VALUES FOR THE THREE FREQUENCY GROUPS
# ============================================================

# Original AOV values are displayed because they are easier to interpret
# in business terms.
#
# The statistical test itself will use Log_AverageOrderValue.

frequency_aov_summary = (
    df_customers
    .groupby("FrequencyGroup")["AverageOrderValue"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

print(frequency_aov_summary)

In [ ]:
# ============================================================
# 9. PREPARE LOG-AOV VALUES FOR THE THREE FREQUENCY GROUPS
# ============================================================

# We use the log-transformed AOV because it is much less skewed
# than the original AverageOrderValue variable.

low_log_aov = df_customers.loc[
    df_customers["FrequencyGroup"] == "Low",
    "Log_AverageOrderValue"
]

medium_log_aov = df_customers.loc[
    df_customers["FrequencyGroup"] == "Medium",
    "Log_AverageOrderValue"
]

high_log_aov = df_customers.loc[
    df_customers["FrequencyGroup"] == "High",
    "Log_AverageOrderValue"
]

print("Low-frequency customers:", len(low_log_aov))
print("Medium-frequency customers:", len(medium_log_aov))
print("High-frequency customers:", len(high_log_aov))

In [ ]:
# ============================================================
# 10. CHECK EQUALITY OF VARIANCES USING LEVENE'S TEST
# ============================================================

# Standard one-way ANOVA assumes that the group variances are approximately equal.
#
# Levene's test checks this assumption.
#
# H0: All three groups have equal variances.
# H1: At least one group has a different variance.
#
# If p > 0.05:
#     Standard one-way ANOVA may be used.
#
# If p < 0.05:
#     Equal-variance assumption is violated,
#     so Welch ANOVA is more appropriate.

levene_stat_3, levene_p_3 = levene(
    low_log_aov,
    medium_log_aov,
    high_log_aov
)

print("Levene statistic:", levene_stat_3)
print("Levene p-value:", levene_p_3)

### Choice of ANOVA Method

Levene's test produced a statistically significant result (p < 0.05),
indicating that the homogeneity of variance assumption required by standard
one-way ANOVA was violated.

Therefore, Welch's ANOVA is used because it allows comparison of mean
customer value across more than two independent groups without requiring
equal variances.

The analysis tests whether mean log-transformed Average Order Value differs
across low-, medium-, and high-frequency customers.

In [ ]:
# ============================================================
# 11. WELCH ANOVA
# ============================================================

# Standard ANOVA is not appropriate because Levene's test showed
# significantly unequal variances across the three frequency groups.
#
# Welch ANOVA is therefore used because it is robust to unequal
# group variances.

from statsmodels.stats.oneway import anova_oneway

welch_anova = anova_oneway(
    df_customers["Log_AverageOrderValue"],
    groups=df_customers["FrequencyGroup"],
    use_var="unequal"
)

print(welch_anova)

In [ ]:
# ============================================================
# 12. MEAN LOG-AOV BY FREQUENCY GROUP
# ============================================================

mean_log_aov_frequency = (
    df_customers
    .groupby("FrequencyGroup")["Log_AverageOrderValue"]
    .mean()
    .round(3)
)

print("Mean Log-Average Order Value:")
print(mean_log_aov_frequency)

In [ ]:
import matplotlib.pyplot as plt

# Mean Average Order Value obtained from the Task 4 analysis
frequency_groups = ["Low\n(1 order)", "Medium\n(2–7 orders)", "High\n(8+ orders)"]
mean_aov = [342.27, 385.79, 426.94]

# Create chart
plt.figure(figsize=(8, 5))

bars = plt.bar(frequency_groups, mean_aov)

# Display the actual mean AOV above each bar
for bar, value in zip(bars, mean_aov):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 6,
        f"£{value:.2f}",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

plt.title("Mean Average Order Value by Purchase Frequency Group",
          fontsize=14, fontweight="bold")

plt.xlabel("Purchase Frequency Group")
plt.ylabel("Mean Average Order Value (£)")

# Start at zero to avoid exaggerating differences
plt.ylim(0, 500)

plt.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUAL CHECK: LOG-AOV DISTRIBUTION BY FREQUENCY GROUP
# ============================================================

import matplotlib.pyplot as plt

group_order = ["Low", "Medium", "High"]

data_to_plot = [
    df_customers.loc[
        df_customers["FrequencyGroup"] == group,
        "Log_AverageOrderValue"
    ]
    for group in group_order
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    data_to_plot,
    tick_labels=[
        "Low\n(1 order)",
        "Medium\n(2–7 orders)",
        "High\n(8+ orders)"
    ]
)

plt.title(
    "Log Average Order Value by Purchase Frequency Group",
    fontweight="bold"
)

plt.xlabel("Purchase Frequency Group")
plt.ylabel("Log(1 + Average Order Value)")

plt.grid(axis="y", alpha=0.20)
plt.tight_layout()
plt.show()